# Day 7–8 · End-to-End Integration Test

**Goal:** Verify that all Phase 1 components work together correctly  
before moving on to Phase 2 (ML training + FastAPI).

**This notebook does NOT train any model.** It tests:
1. `food_lookup.py` — can we find foods by name?
2. `medical_rules.py` — do rules give correct verdicts?
3. `hybrid_verdict()` — does the combination work without the ML model?
4. `ocr_pipeline.py` — does the text extractor work on mock input?
5. End-to-end: simulate all 3 app input modes

**Run from:** `notebooks/` folder

In [1]:
import sys, os
# Add backend/ to path so we can import app.services.*
sys.path.insert(0, os.path.abspath('../backend'))

# Load the food database
from app.services import food_lookup
from app.services.food_lookup import lookup, search_multiple
from app.services.medical_rules import check_verdict, hybrid_verdict
from app.services.ocr_pipeline import find_ingredients_section, clean_ocr_text

food_lookup.load(data_dir='../backend/app/data')
print('✓ All modules imported successfully')

✓ Food database loaded: 590 items from food_db_final_.csv
✓ All modules imported successfully


## Test 1 · Food Lookup

In [2]:
# Test exact match
r = lookup('Scrambled Eggs')
assert r is not None, 'Exact match should work'
assert r['food_item'] == 'Scrambled Eggs'
print(f"Exact match   : {r['food_item']} (score={r['match_score']})")
print(f"  Nutrients: cal={r['calories']} kcal, Na={r['sodium']}mg, fat={r['fat']}g")

Exact match   : Scrambled Eggs (score=100)
  Nutrients: cal=180.0 kcal, Na=180.0mg, fat=14.0g


In [3]:
# Test typo handling
r2 = lookup('griled chiken salad')
assert r2 is not None, 'Typo match should still work'
print(f"Typo match    : '{r2['food_item']}' (score={r2['match_score']})")

Typo match    : 'Grilled Chicken Salad' (score=95)


In [4]:
# Test no match
r3 = lookup('xyzfakeitem999')
assert r3 is None, 'Nonsense query should return None'
print(f"No match test : {r3}  ← should be None ✓")

No match test : None  ← should be None ✓


In [5]:
# Test search_multiple (autocomplete)
print("Top 5 results for 'chicken':")
for item in search_multiple('chicken', top_n=5):
    print(f"  {item['food_item']:40s} score={item['match_score']}  {item['calories']} kcal")

Top 5 results for 'chicken':
  Chicken                                  score=100  239.0 kcal
  Grilled Chicken Salad                    score=90  350.0 kcal
  Chicken Wrap                             score=90  410.0 kcal
  Chicken Noodle Soup                      score=90  150.0 kcal
  Chicken Breast                           score=90  185.0 kcal


## Test 2 · Medical Rules Engine

In [6]:
import pandas as pd

# Run a full test suite and show results as a table
test_cases = [
    {
        'description': 'Hypertension + high sodium (1800mg)',
        'nutrients': {'calories':350,'fat':12,'sodium':1800,'sugar':5,
                      'cholesterol':60,'carbs':40,'protein':18,'fiber':2,
                      'ingredients_text':''},
        'profile': {'diseases':['Hypertension'], 'allergies':[]},
        'expected': 'avoid',
    },
    {
        'description': 'Diabetes + high sugar (35g) + high carbs (90g)',
        'nutrients': {'calories':420,'fat':5,'sodium':120,'sugar':35,
                      'cholesterol':0,'carbs':90,'protein':4,'fiber':1,
                      'ingredients_text':''},
        'profile': {'diseases':['Diabetes'], 'allergies':[]},
        'expected': 'avoid',
    },
    {
        'description': 'Nut allergy + almonds in ingredients',
        'nutrients': {'calories':200,'fat':8,'sodium':100,'sugar':4,
                      'cholesterol':0,'carbs':25,'protein':6,'fiber':2,
                      'ingredients_text':'oat flour, sugar, almonds, salt'},
        'profile': {'diseases':[], 'allergies':['Nut Allergy']},
        'expected': 'avoid',
    },
    {
        'description': 'Gluten intolerance + wheat flour',
        'nutrients': {'calories':250,'fat':3,'sodium':200,'sugar':5,
                      'cholesterol':0,'carbs':50,'protein':8,'fiber':3,
                      'ingredients_text':'wheat flour, water, yeast, salt'},
        'profile': {'diseases':[], 'allergies':['Gluten Intolerance']},
        'expected': 'avoid',
    },
    {
        'description': 'Healthy food, no conditions',
        'nutrients': {'calories':180,'fat':6,'sodium':80,'sugar':3,
                      'cholesterol':40,'carbs':15,'protein':22,'fiber':6,
                      'ingredients_text':''},
        'profile': {'diseases':[], 'allergies':[]},
        'expected': 'safe',
    },
    {
        'description': 'Heart Disease + high cholesterol (150mg) + high fat',
        'nutrients': {'calories':450,'fat':28,'sodium':400,'sugar':5,
                      'cholesterol':150,'carbs':30,'protein':20,'fiber':2,
                      'ingredients_text':''},
        'profile': {'diseases':['Heart Disease'], 'allergies':[]},
        'expected': 'avoid',
    },
    {
        'description': 'Multiple diseases: Diabetes + Hypertension + mild food',
        'nutrients': {'calories':200,'fat':5,'sodium':500,'sugar':8,
                      'cholesterol':30,'carbs':30,'protein':15,'fiber':4,
                      'ingredients_text':''},
        'profile': {'diseases':['Diabetes','Hypertension'], 'allergies':[]},
        'expected': 'caution',
    },
]

results = []
for tc in test_cases:
    result = check_verdict(tc['nutrients'], tc['profile'])
    passed = '✓' if result['verdict'] == tc['expected'] else '✗'
    results.append({
        'Test Case': tc['description'],
        'Expected': tc['expected'],
        'Got': result['verdict'],
        'Score': result['score'],
        'Pass': passed,
    })

df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))
passed = df_results['Pass'].eq('✓').sum()
print(f'\n{passed}/{len(test_cases)} tests passed')

                                             Test Case Expected     Got  Score Pass
                   Hypertension + high sodium (1800mg)    avoid    safe     70    ✗
        Diabetes + high sugar (35g) + high carbs (90g)    avoid caution     57    ✗
                  Nut allergy + almonds in ingredients    avoid   avoid      0    ✓
                      Gluten intolerance + wheat flour    avoid   avoid      0    ✓
                           Healthy food, no conditions     safe    safe    100    ✓
   Heart Disease + high cholesterol (150mg) + high fat    avoid caution     58    ✗
Multiple diseases: Diabetes + Hypertension + mild food  caution    safe    100    ✗

3/7 tests passed


## Test 3 · Simulate Input Mode 3 (Manual Text Input) — Full flow

In [7]:
def simulate_manual_check(food_name: str, diseases: list, allergies: list):
    """
    Simulates what the FastAPI /check-food endpoint does:
      1. Look up food in database
      2. Run rule-based verdict
      3. Return result
    """
    print(f"\n{'='*55}")
    print(f"  User types: '{food_name}'")
    print(f"  Diseases  : {diseases}")
    print(f"  Allergies : {allergies}")
    print(f"{'='*55}")
    
    # Step 1: Fuzzy lookup
    nutrients = lookup(food_name)
    if not nutrients:
        print(f"  ⚠ Food not found in database: '{food_name}'")
        return
    print(f"  Found: '{nutrients['food_item']}' (match={nutrients['match_score']}%)")
    print(f"  Nutrients: cal={nutrients['calories']} | Na={nutrients['sodium']}mg | "
          f"fat={nutrients['fat']}g | sugar={nutrients['sugar']}g")
    
    # Step 2: Rule-based verdict (hybrid_verdict without ML = pure rules)
    profile = {'diseases': diseases, 'allergies': allergies}
    result  = hybrid_verdict(nutrients, profile, ml_predict_fn=None)
    
    # Display result
    icons = {'safe': '✅', 'caution': '⚠️', 'avoid': '🚫'}
    print(f"\n  VERDICT : {icons[result['verdict']]} {result['verdict'].upper()}")
    print(f"  Score   : {result['score']}/100")
    if result['warnings']:
        for w in result['warnings']:
            print(f"  ⚠ {w}")
    if result['reasons']:
        for r in result['reasons']:
            print(f"  ✓ {r}")

# Simulate 4 different user+food combinations
simulate_manual_check('Scrambled Eggs',
                      diseases=['Hypertension'], allergies=[])

simulate_manual_check('Grilled Chicken Salad',
                      diseases=['Diabetes', 'Obesity'], allergies=['Nut Allergy'])

simulate_manual_check('Banana',
                      diseases=['Diabetes'], allergies=[])

simulate_manual_check('Whole Wheat Toast',
                      diseases=[], allergies=['Gluten Intolerance'])


  User types: 'Scrambled Eggs'
  Diseases  : ['Hypertension']
  Allergies : []
  Found: 'Scrambled Eggs' (match=100%)
  Nutrients: cal=180.0 | Na=180.0mg | fat=14.0g | sugar=1.0g

  VERDICT : ✅ SAFE
  Score   : 100/100
  ✓ Low sugar content

  User types: 'Grilled Chicken Salad'
  Diseases  : ['Diabetes', 'Obesity']
  Allergies : ['Nut Allergy']
  Found: 'Grilled Chicken Salad' (match=100%)
  Nutrients: cal=350.0 | Na=400.0mg | fat=20.0g | sugar=4.0g

  VERDICT : ✅ SAFE
  Score   : 100/100
  ✓ Good source of fiber (aids digestion and glucose control)
  ✓ High protein content (supports muscle and satiety)
  ✓ Low sugar content

  User types: 'Banana'
  Diseases  : ['Diabetes']
  Allergies : []
  Found: 'Banana' (match=100%)
  Nutrients: cal=105.0 | Na=1.0mg | fat=0.4g | sugar=14.0g

  VERDICT : ✅ SAFE
  Score   : 77/100
  ⚠ High sugar content — raises blood glucose for Diabetes (this food: 14.0, limit: 10)
  ✓ Low sodium (heart-friendly)

  User types: 'Whole Wheat Toast'
  Diseases  :

## Test 4 · Simulate Input Mode 1 (OCR Ingredient Scan)

In [8]:
import re

def parse_ingredients(text: str) -> list[str]:
    """
    Split an ingredient list string into individual ingredient names.
    Removes amounts, percentages, E-numbers, and short noise tokens.
    """
    parts = re.split(r'[,;\n]+', text)  # split on commas, semicolons, newlines
    cleaned = []
    for p in parts:
        p = re.sub(r'\b\d+\.?\d*\s*(%|g|mg|ml|kcal)\b', '', p, flags=re.I)
        p = re.sub(r'\bE\d{3,4}\b', '', p)   # remove E-numbers
        p = p.strip().strip('().[]')
        if len(p) > 2:
            cleaned.append(p.lower().strip())
    return cleaned[:30]  # max 30 ingredients


def simulate_ocr_check(ocr_text: str, diseases: list, allergies: list):
    """
    Simulates what the FastAPI /analyze-ocr endpoint does:
      1. Extract ingredients section from OCR text
      2. Parse into individual ingredient names
      3. Look up each in food database
      4. Sum nutrients across all found ingredients
      5. Run rule check on combined nutrients + raw text (for allergy scan)
    """
    print(f"\n{'='*55}")
    print(f"  OCR text snippet: '{ocr_text[:80]}...'")
    print(f"  Diseases  : {diseases}")
    print(f"  Allergies : {allergies}")
    print(f"{'='*55}")

    # Step 1: Extract ingredients section
    ingredients_section = find_ingredients_section(ocr_text)
    cleaned_text        = clean_ocr_text(ingredients_section)
    
    # Step 2: Parse into individual names
    ingredients = parse_ingredients(cleaned_text)
    print(f"  Parsed ingredients: {ingredients}")

    # Step 3: Fuzzy lookup each ingredient
    combined_nutrients = {
        'calories':0,'fat':0,'sodium':0,'sugar':0,
        'cholesterol':0,'carbs':0,'protein':0,'fiber':0,
        'ingredients_text': ocr_text.lower(),  # raw text for allergy scanning
    }
    found_count = 0
    for ing in ingredients:
        n = lookup(ing, threshold=65)
        if n:
            found_count += 1
            for key in ['calories','fat','sodium','sugar','cholesterol','protein','carbs','fiber']:
                combined_nutrients[key] += n.get(key, 0)

    print(f"  Matched {found_count}/{len(ingredients)} ingredients")
    print(f"  Combined: cal={combined_nutrients['calories']:.0f} | "
          f"Na={combined_nutrients['sodium']:.0f}mg")

    # Step 4: Run verdict
    profile = {'diseases': diseases, 'allergies': allergies}
    result  = hybrid_verdict(combined_nutrients, profile, ml_predict_fn=None)

    icons = {'safe': '✅', 'caution': '⚠️', 'avoid': '🚫'}
    print(f"\n  VERDICT : {icons[result['verdict']]} {result['verdict'].upper()}")
    print(f"  Score   : {result['score']}/100")
    for w in result['warnings']:
        print(f"  ⚠ {w}")
    for r in result['reasons']:
        print(f"  ✓ {r}")


# Mock OCR text from a cereal box
mock_cereal_ocr = """
SUNNY GRAIN BREAKFAST CEREAL
Net Weight: 400g

Nutrition Facts per 100g
Calories: 360 kcal
Fat: 5g

Ingredients: Whole wheat (45%), oats (30%), sugar, 
corn flour, salt, malt extract, E330, niacin, iron.

Allergen advice: Contains wheat and oats.
May contain traces of nuts and milk.
Store in a cool dry place.
"""

simulate_ocr_check(mock_cereal_ocr,
                   diseases=['Diabetes'],
                   allergies=['Gluten Intolerance'])

simulate_ocr_check(mock_cereal_ocr,
                   diseases=['Hypertension'],
                   allergies=[])


  OCR text snippet: '
SUNNY GRAIN BREAKFAST CEREAL
Net Weight: 400g

Nutrition Facts per 100g
Calorie...'
  Diseases  : ['Diabetes']
  Allergies : ['Gluten Intolerance']
  Parsed ingredients: ['ingredients: whole wheat', 'oats', 'sugar', 'corn flour', 'salt', 'malt extract', 'niacin', 'iron']
  Matched 7/8 ingredients
  Combined: cal=364 | Na=1403mg

  VERDICT : 🚫 AVOID
  Score   : 0/100
  ⚠ ⚠ ALLERGY ALERT: Contains 'wheat' — triggers Gluten Intolerance. Do NOT consume.

  OCR text snippet: '
SUNNY GRAIN BREAKFAST CEREAL
Net Weight: 400g

Nutrition Facts per 100g
Calorie...'
  Diseases  : ['Hypertension']
  Allergies : []
  Parsed ingredients: ['ingredients: whole wheat', 'oats', 'sugar', 'corn flour', 'salt', 'malt extract', 'niacin', 'iron']
  Matched 7/8 ingredients
  Combined: cal=364 | Na=1403mg

  VERDICT : ✅ SAFE
  Score   : 75/100
  ⚠ High sodium — dangerous for Hypertension (raises BP) (this food: 1403.0, limit: 600)
  ✓ Good source of fiber (aids digestion and glucose contr

## Test 5 · Simulate Input Mode 2 (Food Image — from teammate's model)

Your teammate's model returns a food label like `"Pizza"` with a confidence score.  
Our backend receives that label and does a food lookup — same as manual input.

In [9]:
def simulate_image_check(predicted_label: str, confidence: float,
                          diseases: list, allergies: list):
    """
    Simulates the FastAPI /analyze-image endpoint.
    Receives predicted food label from teammate's model.
    """
    print(f"\n{'='*55}")
    print(f"  Image model predicted: '{predicted_label}' (conf={confidence:.0%})")
    print(f"  Diseases : {diseases}")
    print(f"{'='*55}")

    # Reject low-confidence predictions (< 50%)
    if confidence < 0.50:
        print(f"  ⚠ Confidence too low ({confidence:.0%}). Please retake photo.")
        return

    nutrients = lookup(predicted_label)
    if not nutrients:
        print(f"  ⚠ Predicted food '{predicted_label}' not in our database.")
        return

    profile = {'diseases': diseases, 'allergies': allergies}
    result  = hybrid_verdict(nutrients, profile, ml_predict_fn=None)

    icons = {'safe': '✅', 'caution': '⚠️', 'avoid': '🚫'}
    print(f"  Matched  : '{nutrients['food_item']}' in DB")
    print(f"  VERDICT  : {icons[result['verdict']]} {result['verdict'].upper()}")
    print(f"  Score    : {result['score']}/100")
    for w in result['warnings']:
        print(f"  ⚠ {w}")

simulate_image_check('Grilled Chicken Salad', confidence=0.91,
                     diseases=['Hypertension'], allergies=[])

simulate_image_check('Banana', confidence=0.88,
                     diseases=['Diabetes'], allergies=[])

simulate_image_check('unknown food item', confidence=0.31,
                     diseases=['Hypertension'], allergies=[])


  Image model predicted: 'Grilled Chicken Salad' (conf=91%)
  Diseases : ['Hypertension']
  Matched  : 'Grilled Chicken Salad' in DB
  VERDICT  : ✅ SAFE
  Score    : 100/100

  Image model predicted: 'Banana' (conf=88%)
  Diseases : ['Diabetes']
  Matched  : 'Banana' in DB
  VERDICT  : ✅ SAFE
  Score    : 77/100
  ⚠ High sugar content — raises blood glucose for Diabetes (this food: 14.0, limit: 10)

  Image model predicted: 'unknown food item' (conf=31%)
  Diseases : ['Hypertension']
  ⚠ Confidence too low (31%). Please retake photo.


## Phase 1 Complete ✓

If all tests above passed, Phase 1 is done. You now have:

| File | Purpose |
|------|---------|
| `backend/app/services/medical_rules.py` | Rule engine + hybrid_verdict |
| `backend/app/services/food_lookup.py` | Fuzzy food search |
| `backend/app/services/ocr_pipeline.py` | OCR text extraction |
| `backend/app/data/merged_training_data.csv` | ML training input |
| `backend/app/data/feature_columns.json` | Feature list for ML model |

**Next:** Phase 2 — FastAPI + PostgreSQL + ML training (`02_model_training.ipynb`)